# 🧬 Pipeline scRNA-seq — IBD Colon/Recto - GSE214695

## Clasificación Machine Learning

**Objetivo:** entrenar clasificadores (HC vs UC vs CD) con las features pseudobulk
significativas, midiendo si el rendimiento
es mejor que el azar (test de permutación) — exploratorio, no concluyente, con n=18.



# · Importaciones y configuración


In [ ]:
# Montar Google Drive y preparar el gestor de entornos Conda
from google.colab import drive
drive.mount('/content/drive')

!pip install -q condacolab
import condacolab
condacolab.install()

Mounted at /content/drive
⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.11.0-1/Miniforge3-25.11.0-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:08
🔁 Restarting kernel...


In [ ]:
# Instalación del entorno Conda propio de esta fase del proyecto (environment_deg_ml.yaml)

!conda env update -n base -f /content/drive/MyDrive/TFM_IBD_GSE214695/environment_deg_ml.yaml -q

Retrieving notices: ...working... done
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Installing pip dependencies: ...working... done


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from sklearn.preprocessing import label_binarize
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.model_selection import (
    LeaveOneOut, StratifiedKFold, GridSearchCV, cross_val_predict,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score, roc_curve, confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.inspection import permutation_importance
import xgboost as xgb
import shap

print("Librerias cargadas. ")


Librerias cargadas. 


# · Configuración de rutas

In [ ]:
REPO_ROOT = Path("/content/drive/MyDrive/TFM_IBD_GSE214695")
with open(REPO_ROOT / "config" / "params.yaml") as f:
    PARAMS = yaml.safe_load(f)

PROJECT_ROOT = os.environ.get(PARAMS["project_root_env_var"], PARAMS["project_root_default"])

TABLES_08_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['tables'].get('08_deg', 'reports/tables/08_deg')}"
FEATURES_WIDE_PATH = f"{TABLES_08_DIR}/features_wide_table.csv"
FEATURES_META_PATH = f"{TABLES_08_DIR}/features_wide_table_metadata.csv"

OUTPUT_DIR  = f"{PROJECT_ROOT}/{PARAMS['paths']['interim'].get('09_ml', 'data/interim/09_ml')}"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

TABLES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['tables'].get('09_ml', 'reports/tables/09_ml')}"
Path(TABLES_DIR).mkdir(parents=True, exist_ok=True)

FIGURES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['figures'].get('09_ml', 'reports/figures/09_ml')}"
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

PALETTE = {'HC': '#2E86AB', 'UC': '#E84855', 'CD': '#F4A261'}
RANDOM_STATE = 42

# · Carga y preprocesado


In [ ]:
features = pd.read_csv(FEATURES_WIDE_PATH)
meta_cols = pd.read_csv(FEATURES_META_PATH)

assert 'paciente' in features.columns, "Falta la columna 'paciente' en features_wide_table.csv"

# Parseo de condicion desde el propio identificador de paciente
# (formato GSM..._{HC|UC|CD}-{n})

features['condition'] = features['paciente'].str.split('_').str[1].str.split('-').str[0]
valid_conditions = {'HC', 'UC', 'CD'}
bad_rows = ~features['condition'].isin(valid_conditions)
assert not bad_rows.any(), f"No se pudo parsear la condicion de: {features.loc[bad_rows, 'paciente'].tolist()}"

feature_cols = [c for c in features.columns if c not in ('paciente', 'condition')]
n_patients = features.shape[0]
n_features = len(feature_cols)

print(f"Pacientes: {n_patients} | Features: {n_features:,}")
print(f"Ratio features/pacientes: {n_features/n_patients:.0f}:1 ")
print(f"\nDistribucion de condicion:\n{features['condition'].value_counts().to_string()}")


Pacientes: 18 | Features: 5,080
Ratio features/pacientes: 282:1 

Distribucion de condicion:
condition
HC    6
UC    6
CD    6


# · Analisis de NaN

In [ ]:
nan_per_patient = features[feature_cols].isna().sum(axis=1)
nan_per_patient_df = pd.DataFrame({
    'paciente': features['paciente'], 'condition': features['condition'],
    'n_nan': nan_per_patient, 'pct_nan': 100 * nan_per_patient / n_features
})
print("NaN por paciente:")
print(nan_per_patient_df.sort_values('pct_nan', ascending=False).to_string(index=False))

# Comprobación de NaN relacionados con la condicion por tipo celular
meta_indexed = meta_cols.set_index('column_name')
nan_by_ct_cond = []
for ct in meta_cols['tipo_celular'].unique():
    cols_ct = meta_indexed[meta_indexed['tipo_celular'] == ct].index
    cols_ct = [c for c in cols_ct if c in features.columns]
    if not cols_ct:
        continue
    missing_ct = features[cols_ct].isna().all(axis=1)
    for cond in ['HC', 'UC', 'CD']:
        mask_cond = features['condition'] == cond
        if mask_cond.sum() == 0:
            continue
        nan_by_ct_cond.append({
            'tipo_celular': ct, 'condition': cond,
            'pct_pacientes_sin_este_tipo': 100 * missing_ct[mask_cond].mean(),
        })

nan_by_ct_cond_df = pd.DataFrame(nan_by_ct_cond)
pivot_nan = nan_by_ct_cond_df.pivot(index='tipo_celular', columns='condition',
                                     values='pct_pacientes_sin_este_tipo').fillna(0)
print("\n% de pacientes sin pseudobulk valido para cada tipo celular, por condicion:")
print(pivot_nan.round(0).to_string())

# Empleo de 30 puntos de diferencia como umbral para decir que "hay patron"
# Selección propia, no hay una regla fija para esto

NAN_PATTERN_THRESHOLD_PP = 30
diff_range = pivot_nan.max(axis=1) - pivot_nan.min(axis=1)
ct_with_pattern = diff_range[diff_range >= NAN_PATTERN_THRESHOLD_PP].index.tolist()

USE_MISSING_INDICATOR = True
if len(ct_with_pattern) > 0:
    print(f"\n  Patron de NaN relacionado con la condicion en: {ct_with_pattern} "
          f"(diferencia >= {NAN_PATTERN_THRESHOLD_PP} puntos porcentuales entre grupos).")
else:
    print(f"\nNo se detecta un patron de NaN claramente relacionado con la condicion ")


NaN por paciente:
       paciente condition  n_nan   pct_nan
GSM6614354_UC-1        UC   1313 25.846457
GSM6614350_HC-3        HC    638 12.559055
GSM6614357_UC-4        UC    188  3.700787
GSM6614358_UC-5        UC    188  3.700787
GSM6614353_HC-6        HC      6  0.118110
GSM6614349_HC-2        HC      5  0.098425
GSM6614360_CD-1        CD      5  0.098425
GSM6614351_HC-4        HC      2  0.039370
GSM6614348_HC-1        HC      1  0.019685
GSM6614352_HC-5        HC      1  0.019685
GSM6614361_CD-2        CD      1  0.019685
GSM6614365_CD-6        CD      1  0.019685
GSM6614356_UC-3        UC      0  0.000000
GSM6614355_UC-2        UC      0  0.000000
GSM6614359_UC-6        UC      0  0.000000
GSM6614362_CD-3        CD      0  0.000000
GSM6614363_CD-4        CD      0  0.000000
GSM6614364_CD-5        CD      0  0.000000

% de pacientes sin pseudobulk valido para cada tipo celular, por condicion:
condition                             CD     HC    UC
tipo_celular                      

# · Deteccion de outliers

Un paciente es candidato a outlier si su correlacion mediana con el resto de pacientes
de su misma condicion es notablemente mas baja que la del resto de su grupo.


In [ ]:
X_diag = features[feature_cols].copy()
X_diag = X_diag.fillna(X_diag.median())

corr_matrix = X_diag.T.corr()  # correlacion entre pacientes (transponer: genes en filas)
corr_matrix.index = features['paciente'].values
corr_matrix.columns = features['paciente'].values

outlier_scores = []
for i, row in features.iterrows():
    pid, cond = row['paciente'], row['condition']
    same_cond_patients = features.loc[features['condition'] == cond, 'paciente']
    same_cond_patients = [p for p in same_cond_patients if p != pid]
    if not same_cond_patients:
        continue
    median_corr = corr_matrix.loc[pid, same_cond_patients].median()
    outlier_scores.append({'paciente': pid, 'condition': cond, 'median_corr_same_condition': median_corr})

outlier_df = pd.DataFrame(outlier_scores)
outlier_flags = []
for cond in ['HC', 'UC', 'CD']:
    sub = outlier_df[outlier_df['condition'] == cond]
    q1, q3 = sub['median_corr_same_condition'].quantile([0.25, 0.75])
    iqr = q3 - q1
    threshold = q1 - 1.5 * iqr
    flagged = sub[sub['median_corr_same_condition'] < threshold]['paciente'].tolist()
    outlier_flags.extend(flagged)

print("Correlacion mediana con pacientes de la misma condicion:")
print(outlier_df.sort_values('median_corr_same_condition').to_string(index=False))

OUTLIERS_TO_TEST = outlier_flags if outlier_flags else []
print(f"\nPacientes que se probaran como exclusion en la seccion de robustez: {OUTLIERS_TO_TEST}")


Correlacion mediana con pacientes de la misma condicion:
       paciente condition  median_corr_same_condition
GSM6614364_CD-5        CD                    0.774350
GSM6614354_UC-1        UC                    0.780031
GSM6614359_UC-6        UC                    0.805525
GSM6614360_CD-1        CD                    0.815210
GSM6614356_UC-3        UC                    0.821408
GSM6614348_HC-1        HC                    0.822547
GSM6614358_UC-5        UC                    0.827593
GSM6614357_UC-4        UC                    0.827593
GSM6614361_CD-2        CD                    0.828890
GSM6614355_UC-2        UC                    0.835444
GSM6614362_CD-3        CD                    0.840589
GSM6614363_CD-4        CD                    0.847437
GSM6614365_CD-6        CD                    0.849033
GSM6614351_HC-4        HC                    0.854336
GSM6614349_HC-2        HC                    0.855828
GSM6614350_HC-3        HC                    0.855828
GSM6614353_HC-6        HC

# · Pipeline y validación

Todo el preprocesado (imputar, escalar, seleccionar features) se ajusta con los datos de entrenamiento de cada partición, no con todos los pacientes a la vez.


In [ ]:
VARIANCE_THRESHOLD = 0.0  # elimina solo columnas con varianza 0

SELECT_K_BEST_THRESHOLD = 100  # si tras VarianceThreshold quedan >100 features, añade SelectKBest
INNER_CV_FOLDS = 3  # k-fold estratificado pequeño para la CV interna

NEEDS_SCALING = {'logreg': True, 'svm': True, 'rf': False, 'xgb': False}

PARAM_GRIDS = {
    'logreg': {
        'clf__C': [0.01, 0.1, 1, 10],
        'clf__penalty': ['l1', 'l2'],
    },
    'rf': {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [2, 3],
        'clf__min_samples_leaf': [2, 3],
    },
    'svm': {
        'clf__C': [0.01, 0.1, 1],
    },
    'xgb': {
        'clf__n_estimators': [50, 100],
        'clf__learning_rate': [0.01, 0.05],
        'clf__max_depth': [2, 3],
        'clf__subsample': [0.8, 1.0],
    },
}


def make_base_estimator(name, class_weight_map=None, scale_pos_weight=None, n_classes=2):
    if name == 'logreg':
        return LogisticRegression(solver='saga', max_iter=5000,
                                   class_weight=class_weight_map, random_state=RANDOM_STATE)
    elif name == 'rf':
        return RandomForestClassifier(class_weight=class_weight_map, random_state=RANDOM_STATE)
    elif name == 'svm':
        return SVC(kernel='linear', probability=True, class_weight=class_weight_map,
                    random_state=RANDOM_STATE)
    elif name == 'xgb':
        kwargs = dict(random_state=RANDOM_STATE, eval_metric='logloss' if n_classes == 2 else 'mlogloss')
        if n_classes == 2 and scale_pos_weight is not None:
            kwargs['scale_pos_weight'] = scale_pos_weight
        return xgb.XGBClassifier(**kwargs)
    raise ValueError(name)

def build_pipeline(name, n_features_in, add_indicator, class_weight_map=None,
                    scale_pos_weight=None, n_classes=2):
    steps = [
        ('imputer', SimpleImputer(strategy='median', add_indicator=add_indicator)),
        ('vt', VarianceThreshold(threshold=VARIANCE_THRESHOLD)),
    ]
    if NEEDS_SCALING[name]:
        steps.append(('scaler', StandardScaler()))
    if n_features_in > SELECT_K_BEST_THRESHOLD:
        # k de SelectKBest se deja como hiperparametro fijo
        steps.append(('kbest', SelectKBest(score_func=f_classif, k=min(50, n_features_in))))
    steps.append(('clf', make_base_estimator(name, class_weight_map, scale_pos_weight, n_classes)))
    return Pipeline(steps)

print("Utilidades de pipeline definidas")


Utilidades de pipeline definidas


In [ ]:
def bootstrap_ci(y_true, y_score, metric_fn, n_boot=2000, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    scores = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            scores.append(metric_fn(y_true[idx], y_score[idx]))
        except ValueError:
            continue
    lo, hi = np.nanpercentile(scores, [2.5, 97.5])
    return float(np.nanmean(scores)), float(lo), float(hi)

def run_nested_loocv(X, y, name, add_indicator, class_weight_map=None,
                      scale_pos_weight=None, n_classes=2, needs_proba=True):

    # LOOCV por fuera, y dentro de cada particion se buscan los mejores hiperparametros con 3-fold
    # Todo el preprocesado esta dentro del Pipeline, asi que se recalcula cada vez solo con
    # los datos de entrenamiento de esa particion

    outer_cv = LeaveOneOut()
    inner_cv = StratifiedKFold(n_splits=INNER_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    pipe = build_pipeline(name, X.shape[1], add_indicator, class_weight_map, scale_pos_weight, n_classes)
    grid = PARAM_GRIDS[name]
    search = GridSearchCV(pipe, grid, cv=inner_cv,
                           scoring='roc_auc' if n_classes == 2 else 'f1_macro',
                           n_jobs=1)

    method = 'predict_proba' if needs_proba else 'predict'
    preds = cross_val_predict(search, X, y, cv=outer_cv, method=method, n_jobs=-1)
    search.fit(X, y)
    best_params = search.best_params_
    return preds, best_params


def permutation_test(fixed_pipe, X, y, metric_fn, needs_proba,
                                 n_permutations=1000, seed=RANDOM_STATE):

    # Se decidió no usar la funcion de sklearn (permutation_test_score)
    # porque calcula la metrica por particion, y con LOOCV eso no funciona bien
    # (cada particion deja solo 1 caso de test). Aqui, en cada permutacion, se
    # recalculan las 18 predicciones de LOOCV y se junta todo en una sola metrica,
    # igual que se hace con los datos reales.
    # cambiar n_permutations a 100 si va muy lento

    outer_cv = LeaveOneOut()
    method = 'predict_proba' if needs_proba else 'predict'

    preds_true = cross_val_predict(fixed_pipe, X, y, cv=outer_cv, method=method, n_jobs=-1)
    score_true = metric_fn(y, preds_true[:, 1] if needs_proba else preds_true)

    rng = np.random.RandomState(seed)
    perm_scores = np.empty(n_permutations)
    for i in range(n_permutations):
        y_perm = rng.permutation(y)
        preds_perm = cross_val_predict(fixed_pipe, X, y_perm, cv=outer_cv, method=method, n_jobs=-1)
        perm_scores[i] = metric_fn(y_perm, preds_perm[:, 1] if needs_proba else preds_perm)

    # suma 1 arriba y abajo para que el p-valor nunca de exactamente 0
    pvalue = (np.sum(perm_scores >= score_true) + 1) / (n_permutations + 1)
    return score_true, perm_scores, pvalue


print("Funciones de evaluacion definidas")


Funciones de evaluacion definidas


# · Comparación Binaria: HC vs IBD (UC+CD)


In [ ]:
le_binary = LabelEncoder()
y_bin_labels = features['condition'].map({'HC': 'HC', 'UC': 'IBD', 'CD': 'IBD'})
y_bin = le_binary.fit_transform(y_bin_labels)  # HC=0, IBD=1 (o al reves; se imprime abajo)
print(f"Clases binarias: {dict(zip(le_binary.classes_, range(len(le_binary.classes_))))}")
print(f"Distribucion: {pd.Series(y_bin_labels).value_counts().to_dict()}")

X_all = features[feature_cols].values
n_hc = (y_bin_labels == 'HC').sum()
n_ibd = (y_bin_labels == 'IBD').sum()

# class_weight='balanced' porque hay el doble de IBD que de HC (12 vs 6)

CLASS_WEIGHT_BIN = 'balanced'
SCALE_POS_WEIGHT_BIN = n_hc / n_ibd  # calculado sobre el dataset completo

results_binary = {}
predictions_binary = {}
best_params_binary = {}

for model_name in ['logreg', 'rf', 'svm', 'xgb']:
    preds, best_params = run_nested_loocv(
        X_all, y_bin, model_name, add_indicator=USE_MISSING_INDICATOR,
        class_weight_map=CLASS_WEIGHT_BIN, scale_pos_weight=SCALE_POS_WEIGHT_BIN,
        n_classes=2, needs_proba=True,
    )
    y_proba = preds[:, 1]
    y_pred_class = (y_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_bin, y_proba)
    f1 = f1_score(y_bin, y_pred_class)
    acc = accuracy_score(y_bin, y_pred_class)
    auc_mean, auc_lo, auc_hi = bootstrap_ci(y_bin, y_proba, roc_auc_score)

    results_binary[model_name] = {
        'auc': auc, 'f1': f1, 'accuracy': acc,
        'auc_ci_lo': auc_lo, 'auc_ci_hi': auc_hi,
    }
    predictions_binary[model_name] = {'y_true': y_bin, 'y_proba': y_proba, 'y_pred': y_pred_class}
    best_params_binary[model_name] = best_params

    print(f"{model_name:8s} | AUC={auc:.3f} [{auc_lo:.3f}, {auc_hi:.3f}] | "
          f"F1={f1:.3f} | Acc={acc:.3f} | mejores hiperparametros: {best_params}")

results_binary_df = pd.DataFrame(results_binary).T
results_binary_df.to_csv(f"{TABLES_DIR}/binary_metrics.csv")


Clases binarias: {'HC': 0, 'IBD': 1}
Distribucion: {'IBD': 12, 'HC': 6}
logreg   | AUC=1.000 [1.000, 1.000] | F1=1.000 | Acc=1.000 | mejores hiperparametros: {'clf__C': 0.01, 'clf__penalty': 'l2'}
rf       | AUC=0.972 [0.870, 1.000] | F1=0.960 | Acc=0.944 | mejores hiperparametros: {'clf__max_depth': 2, 'clf__min_samples_leaf': 2, 'clf__n_estimators': 100}
svm      | AUC=1.000 [1.000, 1.000] | F1=0.960 | Acc=0.944 | mejores hiperparametros: {'clf__C': 0.01}
xgb      | AUC=0.611 [0.200, 1.000] | F1=0.880 | Acc=0.833 | mejores hiperparametros: {'clf__learning_rate': 0.01, 'clf__max_depth': 2, 'clf__n_estimators': 50, 'clf__subsample': 1.0}


In [ ]:
# Matrices de confusion y curvas ROC para la memoria

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, model_name in zip(axes, ['logreg', 'rf', 'svm', 'xgb']):
    p = predictions_binary[model_name]
    cm = confusion_matrix(p['y_true'], p['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=le_binary.classes_).plot(ax=ax, colorbar=False)
    ax.set_title(f"{model_name} (binario)")
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/confusion_matrices_binary.png", dpi=130, bbox_inches='tight', facecolor='white')
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 6))
for model_name in ['logreg', 'rf', 'svm', 'xgb']:
    p = predictions_binary[model_name]
    fpr, tpr, _ = roc_curve(p['y_true'], p['y_proba'])
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={results_binary[model_name]['auc']:.2f})")
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("ROC - Binario (HC vs IBD)")
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/roc_binary.png", dpi=130, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   -> Figuras guardadas: confusion_matrices_binary.png, roc_binary.png")


   -> Figuras guardadas: confusion_matrices_binary.png, roc_binary.png


In [ ]:
# Se elije el mejor modelo binario por AUC (F1 para desempatar)

best_model_binary = results_binary_df['auc'].idxmax()
print(f"Mejor modelo binario: {best_model_binary} (AUC={results_binary_df.loc[best_model_binary, 'auc']:.3f})")

# Test de permutacion
N_PERMUTATIONS = 1000

fixed_params = {k.replace('clf__', ''): v for k, v in best_params_binary[best_model_binary].items()}
fixed_pipe = build_pipeline(best_model_binary, X_all.shape[1], USE_MISSING_INDICATOR,
                             CLASS_WEIGHT_BIN, SCALE_POS_WEIGHT_BIN, n_classes=2)
fixed_pipe.set_params(**{f'clf__{k}': v for k, v in fixed_params.items()})

score_real, perm_scores, pvalue_binary = permutation_test(
    fixed_pipe, X_all, y_bin, metric_fn=roc_auc_score, needs_proba=True,
    n_permutations=N_PERMUTATIONS, seed=RANDOM_STATE,
)

print(f"\nTest de permutacion (binario, {best_model_binary}, {N_PERMUTATIONS} permutaciones, "
      f"hiperparametros fijos={fixed_params}):")
print(f"  AUC real={score_real:.3f} | AUC medio bajo permutacion={np.mean(perm_scores):.3f} | p-valor={pvalue_binary:.4f}")
print(f"  {'Mejor que el azar (p<0.05)' if pvalue_binary < 0.05 else 'NO se distingue claramente del azar'}")


Mejor modelo binario: logreg (AUC=1.000)

Test de permutacion (binario, logreg, 1000 permutaciones, hiperparametros fijos={'C': 0.01, 'penalty': 'l2'}):
  AUC real=1.000 | AUC medio bajo permutacion=0.431 | p-valor=0.0010
  Mejor que el azar (p<0.05)


# · Comparación Multiclase: HC vs UC vs CD

In [ ]:
le_multi = LabelEncoder()
y_multi = le_multi.fit_transform(features['condition'])
print(f"Clases multiclase: {dict(zip(le_multi.classes_, range(len(le_multi.classes_))))}")
print(f"Distribucion: {features['condition'].value_counts().to_dict()}")

class_counts = features['condition'].value_counts()
CLASS_WEIGHT_MULTI = 'balanced' if (class_counts < 5).any() or class_counts.nunique() > 1 else None
print(f"class_weight usado: {CLASS_WEIGHT_MULTI}")

results_multi = {}
predictions_multi = {}
best_params_multi = {}

for model_name in ['logreg', 'rf', 'svm', 'xgb']:

    # XGBoost no tiene scale_pos_weight para multiclase, asi que no se aplica
    # ningun peso extra - con 6/6/6 (o 4/6/6 sin los outliers) no esta muy desbalanceado

    preds_proba, best_params = run_nested_loocv(
        X_all, y_multi, model_name, add_indicator=USE_MISSING_INDICATOR,
        class_weight_map=CLASS_WEIGHT_MULTI, scale_pos_weight=None,
        n_classes=len(le_multi.classes_), needs_proba=True,   # antes: False
    )
    y_pred_class = np.argmax(preds_proba, axis=1)

    f1_macro = f1_score(y_multi, y_pred_class, average='macro')
    acc = accuracy_score(y_multi, y_pred_class)

    results_multi[model_name] = {'f1_macro': f1_macro, 'accuracy': acc}
    predictions_multi[model_name] = {'y_true': y_multi, 'y_pred': y_pred_class, 'y_proba': preds_proba}
    best_params_multi[model_name] = best_params

    print(f"{model_name:8s} | F1_macro={f1_macro:.3f} | Acc={acc:.3f} | "
          f"mejores hiperparametros: {best_params}")

results_multi_df = pd.DataFrame(results_multi).T
results_multi_df.to_csv(f"{TABLES_DIR}/multiclass_metrics.csv")


Clases multiclase: {'CD': 0, 'HC': 1, 'UC': 2}
Distribucion: {'HC': 6, 'UC': 6, 'CD': 6}
class_weight usado: None
logreg   | F1_macro=0.731 | Acc=0.722 | mejores hiperparametros: {'clf__C': 0.01, 'clf__penalty': 'l2'}
rf       | F1_macro=0.784 | Acc=0.778 | mejores hiperparametros: {'clf__max_depth': 2, 'clf__min_samples_leaf': 2, 'clf__n_estimators': 100}
svm      | F1_macro=0.836 | Acc=0.833 | mejores hiperparametros: {'clf__C': 0.01}
xgb      | F1_macro=0.444 | Acc=0.444 | mejores hiperparametros: {'clf__learning_rate': 0.05, 'clf__max_depth': 2, 'clf__n_estimators': 100, 'clf__subsample': 1.0}


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, model_name in zip(axes, ['logreg', 'rf', 'svm', 'xgb']):
    p = predictions_multi[model_name]
    cm = confusion_matrix(p['y_true'], p['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=le_multi.classes_).plot(ax=ax, colorbar=False)
    ax.set_title(f"{model_name} (multiclase)")
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/confusion_matrices_multiclass.png", dpi=130, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   -> Figura guardada: confusion_matrices_multiclass.png")

y_multi_bin = label_binarize(y_multi, classes=list(range(len(le_multi.classes_))))

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, model_name in zip(axes, ['logreg', 'rf', 'svm', 'xgb']):
    y_proba = predictions_multi[model_name]['y_proba']
    for c in range(y_multi_bin.shape[1]):
        fpr, tpr, _ = roc_curve(y_multi_bin[:, c], y_proba[:, c])
        roc_auc_c = roc_auc_score(y_multi_bin[:, c], y_proba[:, c])
        ax.plot(fpr, tpr, label=f"{le_multi.classes_[c]} (AUC={roc_auc_c:.2f})")
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title(f"{model_name} (multiclase, one-vs-rest)")
    ax.legend(fontsize=7)
plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/roc_multiclass.png", dpi=130, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   -> Figura guardada: roc_multiclass.png")

best_model_multi = results_multi_df['f1_macro'].idxmax()
print(f"\nMejor modelo multiclase: {best_model_multi} (F1_macro={results_multi_df.loc[best_model_multi, 'f1_macro']:.3f})")


   -> Figura guardada: confusion_matrices_multiclass.png
   -> Figura guardada: roc_multiclass.png

Mejor modelo multiclase: svm (F1_macro=0.836)


In [ ]:
fixed_params_m = {k.replace('clf__', ''): v for k, v in best_params_multi[best_model_multi].items()}
fixed_pipe_m = build_pipeline(best_model_multi, X_all.shape[1], USE_MISSING_INDICATOR,
                               CLASS_WEIGHT_MULTI, None, n_classes=len(le_multi.classes_))
fixed_pipe_m.set_params(**{f'clf__{k}': v for k, v in fixed_params_m.items()})

from functools import partial
f1_macro_fn = partial(f1_score, average='macro', zero_division=0)

score_real_m, perm_scores_m, pvalue_multi = permutation_test(
    fixed_pipe_m, X_all, y_multi, metric_fn=f1_macro_fn, needs_proba=False,
    n_permutations=N_PERMUTATIONS, seed=RANDOM_STATE,
)

print(f"Test de permutacion (multiclase, {best_model_multi}, {N_PERMUTATIONS} permutaciones):")
print(f"  F1_macro real={score_real_m:.3f} | medio bajo permutacion={np.mean(perm_scores_m):.3f} | p-valor={pvalue_multi:.4f}")
print(f"  {'Mejor que el azar (p<0.05)' if pvalue_multi < 0.05 else 'NO se distingue claramente del azar'}")

# Como solo hay 2 tests (binario y multiclase), se corrije el umbral a 0.025 (0.05/2)

ALPHA_CORRECTED = 0.05 / 2
print(f"\nUmbral corregido (Bonferroni, 2 tests): {ALPHA_CORRECTED}")
print(f"  Binario:    p={pvalue_binary:.4f} -> {'sigue siendo significativo' if pvalue_binary < ALPHA_CORRECTED else 'deja de serlo tras correccion'}")
print(f"  Multiclase: p={pvalue_multi:.4f} -> {'sigue siendo significativo' if pvalue_multi < ALPHA_CORRECTED else 'deja de serlo tras correccion'}")


Test de permutacion (multiclase, svm, 1000 permutaciones):
  F1_macro real=0.892 | medio bajo permutacion=0.240 | p-valor=0.0010
  Mejor que el azar (p<0.05)

Umbral corregido (Bonferroni, 2 tests): 0.025
  Binario:    p=0.0010 -> sigue siendo significativo
  Multiclase: p=0.0010 -> sigue siendo significativo


# · Robustez: con y sin las muestras candidatas a outlier



In [ ]:
DECISION_EXCLUDE_OUTLIERS = False
significant_multi = False

if not OUTLIERS_TO_TEST:
    print("No hay candidatos a outlier calculados, se usan los 18 pacientes.")
    FINAL_PATIENTS_MASK = pd.Series(True, index=features.index)
else:
    mask_keep = ~features['paciente'].isin(OUTLIERS_TO_TEST)
    X_reduced = features.loc[mask_keep, feature_cols].values
    y_bin_reduced = y_bin[mask_keep.values]

    print(f"Version A (n={len(y_bin)}): todos los pacientes")
    print(f"Version B (n={len(y_bin_reduced)}): sin {OUTLIERS_TO_TEST}")

    preds_b, _ = run_nested_loocv(
        X_reduced, y_bin_reduced, best_model_binary, add_indicator=USE_MISSING_INDICATOR,
        class_weight_map=CLASS_WEIGHT_BIN, scale_pos_weight=SCALE_POS_WEIGHT_BIN,
        n_classes=2, needs_proba=True,
    )
    y_pred_b = (preds_b[:, 1] >= 0.5).astype(int)

    idx_common = np.where(mask_keep.values)[0]
    y_pred_a_common = predictions_binary[best_model_binary]['y_pred'][idx_common]
    y_true_common = y_bin[idx_common]

    from statsmodels.stats.contingency_tables import mcnemar
    correct_a = (y_pred_a_common == y_true_common)
    correct_b = (y_pred_b == y_true_common)
    table = pd.crosstab(correct_a, correct_b)

    # Si falta alguna combinacion en la tabla se rellena con 0 para que McNemar no de error

    table = table.reindex(index=[False, True], columns=[False, True], fill_value=0)
    mcnemar_result = mcnemar(table.values, exact=True)
    print(f"\nMcNemar (binario, A vs B, pacientes comunes): estadistico={mcnemar_result.statistic:.3f}, "
          f"p={mcnemar_result.pvalue:.4f}")
    print(f"  {'Diferencia significativa ' if mcnemar_result.pvalue < 0.05 else 'Sin diferencia significativa, se mantienen los 18 pacientes por transparencia'}")

    DECISION_EXCLUDE_OUTLIERS = mcnemar_result.pvalue < 0.05
    FINAL_PATIENTS_MASK = pd.Series(True, index=features.index)  # se ajusta abajo tras el bloque multiclase


Version A (n=18): todos los pacientes
Version B (n=15): sin ['GSM6614348_HC-1', 'GSM6614354_UC-1', 'GSM6614364_CD-5']

McNemar (binario, A vs B, pacientes comunes): estadistico=0.000, p=1.0000
  Sin diferencia significativa, se mantienen los 18 pacientes por transparencia


In [ ]:
# Para multiclase no se puede usar McNemar (es solo para 2 clases), asi que se
# compara con bootstrap la diferencia de F1_macro entre las dos versiones

if OUTLIERS_TO_TEST:
    y_multi_reduced = y_multi[mask_keep.values]
    preds_b_multi, _ = run_nested_loocv(
        X_reduced, y_multi_reduced, best_model_multi, add_indicator=USE_MISSING_INDICATOR,
        class_weight_map=CLASS_WEIGHT_MULTI, scale_pos_weight=None,
        n_classes=len(le_multi.classes_), needs_proba=False,
    )
    y_pred_a_multi_common = predictions_multi[best_model_multi]['y_pred'][idx_common]
    y_true_multi_common = y_multi[idx_common]

    f1_a = f1_score(y_true_multi_common, y_pred_a_multi_common, average='macro')
    f1_b = f1_score(y_true_multi_common, preds_b_multi, average='macro')

    rng = np.random.RandomState(RANDOM_STATE)
    n_common = len(idx_common)
    diffs = []
    for _ in range(2000):
        idx_boot = rng.randint(0, n_common, n_common)
        f1_a_boot = f1_score(y_true_multi_common[idx_boot], y_pred_a_multi_common[idx_boot],
                              average='macro', zero_division=0)
        f1_b_boot = f1_score(y_true_multi_common[idx_boot], preds_b_multi[idx_boot],
                              average='macro', zero_division=0)
        diffs.append(f1_b_boot - f1_a_boot)
    diffs = np.array(diffs)
    ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
    print(f"Diferencia sobre pacientes comunes: {f1_b - f1_a:+.3f}, "
          f"IC 95% bootstrap [{ci_lo:+.3f}, {ci_hi:+.3f}]")
    significant_multi = not (ci_lo <= 0 <= ci_hi)
    print(f"  {'Diferencia significativa' if significant_multi else 'No hay diferencias significativas, se mantienen los 18 pacientes'}")

    EXCLUDE_FINAL = DECISION_EXCLUDE_OUTLIERS or significant_multi
    print(f"\nDECISION FINAL: "
          f"{'excluir ' + str(OUTLIERS_TO_TEST) + ' de los analisis interpretativos' if EXCLUDE_FINAL else 'mantener los 18 pacientes'}")
    FINAL_PATIENTS_MASK = mask_keep if EXCLUDE_FINAL else pd.Series(True, index=features.index)
else:
    FINAL_PATIENTS_MASK = pd.Series(True, index=features.index)


Diferencia sobre pacientes comunes: +0.000, IC 95% bootstrap [-0.208, +0.213]
  No hay diferencias significativas, se mantienen los 18 pacientes

DECISION FINAL: mantener los 18 pacientes


# · Interpretabilidad (SHAP / importancia por permutacion)



In [ ]:
use_final_subset = FINAL_PATIENTS_MASK.sum() < n_patients
X_final = features.loc[FINAL_PATIENTS_MASK, feature_cols].values if use_final_subset else X_all
y_final = y_bin[FINAL_PATIENTS_MASK.values] if use_final_subset else y_bin

final_pipe = build_pipeline(best_model_binary, X_final.shape[1], USE_MISSING_INDICATOR,
                             CLASS_WEIGHT_BIN, SCALE_POS_WEIGHT_BIN, n_classes=2)
final_pipe.set_params(**{f'clf__{k}': v for k, v in fixed_params.items()})
final_pipe.fit(X_final, y_final)

# Se recuperan los nombres de las columnas que quedan tras imputar/filtrar para que el SHAP las etiquete bien
feature_names_arr = np.array(feature_cols)
imputer_step = final_pipe.named_steps['imputer']
if USE_MISSING_INDICATOR and imputer_step.indicator_ is not None:
    missing_idx = imputer_step.indicator_.features_
    names_before_vt = np.concatenate([
        feature_names_arr,
        np.array([f"MISSING__{feature_cols[i]}" for i in missing_idx]),
    ])
else:
    names_before_vt = feature_names_arr

mask_vt = final_pipe.named_steps['vt'].get_support()
names_after_vt = names_before_vt[mask_vt]

if 'kbest' in dict(final_pipe.steps):
    mask_kbest = final_pipe.named_steps['kbest'].get_support()
    feature_names_final = names_after_vt[mask_kbest]
else:
    feature_names_final = names_after_vt

X_transformed = final_pipe[:-1].transform(X_final)
print(f"Modelo final para SHAP: {best_model_binary} | {X_transformed.shape[1]} features tras preprocesado")


Modelo final para SHAP: logreg | 50 features tras preprocesado


In [ ]:
clf_final = final_pipe.named_steps['clf']

if best_model_binary in ('rf', 'xgb'):
    explainer = shap.TreeExplainer(clf_final)
    shap_values = explainer.shap_values(X_transformed)

    # La libreria de SHAP a veces devuelve el resultado como lista [clase0, clase1] y
    # otras veces como un array 3D. Se comprueba cual de los dos es y establecemos la
    # clase positiva

    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # clase positiva (IBD)
    elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]
else:
    # Para SVM o regresion logistica se usa KernelExplainer, como es más lento
    # se reduce el numero de puntos de referencia con shap.kmeans

    if best_model_binary == 'svm':
        print("SVM gana")
    np.random.seed(RANDOM_STATE)
    background = shap.kmeans(X_transformed, min(10, X_transformed.shape[0]))
    explainer = shap.KernelExplainer(clf_final.predict_proba, background)
    shap_values_full = explainer.shap_values(X_transformed, nsamples=100)

    # Mismo problema de formato que antes, se soluciona igual

    if isinstance(shap_values_full, list):
        shap_values = shap_values_full[1]
    elif isinstance(shap_values_full, np.ndarray) and shap_values_full.ndim == 3:
        shap_values = shap_values_full[:, :, 1]
    else:
        shap_values = shap_values_full

shap_importance = pd.DataFrame({
    'feature': feature_names_final,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False)

shap_importance.to_csv(f"{TABLES_DIR}/shap_importance.csv", index=False)
print(shap_importance.head(20).to_string(index=False))


  0%|          | 0/18 [00:00<?, ?it/s]

                                             feature  mean_abs_shap
                        Colonocytes__DPY30__CD_vs_HC       0.013367
                       Colonocytes__CYP3A4__CD_vs_HC       0.013185
                              PC_IgA__TLE1__CD_vs_HC       0.013180
                     Colonocytes__APOBEC3C__CD_vs_HC       0.011820
                           CD4__UBXN10-AS1__UC_vs_HC       0.011329
Monocyte_(subtipo_indeterminado)__PLA2G12A__UC_vs_HC       0.010467
    Monocyte_(subtipo_indeterminado)__PIM1__UC_vs_HC       0.009160
  Monocyte_(subtipo_indeterminado)__S100A8__CD_vs_HC       0.009064
 Monocyte_(subtipo_indeterminado)__CTTNBP2__UC_vs_HC       0.008697
                              PC_IgA__TLE1__UC_vs_HC       0.007577
                        Colonocytes__DPY30__UC_vs_HC       0.007498
                            Goblet__LGALS4__CD_vs_HC       0.007270
    Monocyte_(subtipo_indeterminado)__VCAN__UC_vs_HC       0.007214
                   Epithelium_Ribhi__OLFM4__UC_v

In [ ]:
# Importancia por permutacion, como referencia adicional al SHAP
perm_imp = permutation_importance(final_pipe, X_final, y_final, n_repeats=30,
                                   random_state=RANDOM_STATE, scoring='roc_auc')
perm_imp_df = pd.DataFrame({
    'feature': feature_cols,
    'perm_importance_mean': perm_imp.importances_mean,
    'perm_importance_std': perm_imp.importances_std,
}).sort_values('perm_importance_mean', ascending=False)
perm_imp_df.to_csv(f"{TABLES_DIR}/permutation_importance.csv", index=False)
print("\nTop 20 por importancia de permutacion:")
print(perm_imp_df.head(20).to_string(index=False))



Top 20 por importancia de permutacion:
                  feature  perm_importance_mean  perm_importance_std
PC_IgG__ZSCAN18__UC_vs_HC                   0.0                  0.0
  B_cell__NUDT1__UC_vs_CD                   0.0                  0.0
    CD4__AARSD1__CD_vs_HC                   0.0                  0.0
CD4__AC004585.1__UC_vs_HC                   0.0                  0.0
CD4__AC005224.3__CD_vs_HC                   0.0                  0.0
CD4__AC005224.3__UC_vs_HC                   0.0                  0.0
CD4__AC017002.3__CD_vs_HC                   0.0                  0.0
CD4__AC017002.3__UC_vs_HC                   0.0                  0.0
 PC_IgG__WNT10A__UC_vs_HC                   0.0                  0.0
  PC_IgG__WIPF1__UC_vs_HC                   0.0                  0.0
  PC_IgG__WIPF1__UC_vs_CD                   0.0                  0.0
   PC_IgG__WHRN__UC_vs_HC                   0.0                  0.0
 PC_IgG__WASHC3__UC_vs_HC                   0.0                

In [ ]:
# Graficos de SHAP (resumen top 20, beeswarm, barras)

try:
    plt.figure()
    shap.summary_plot(shap_values, X_transformed, feature_names=feature_names_final,
                       max_display=20, show=False)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/shap_summary_top20.png", dpi=130, bbox_inches='tight', facecolor='white')
    plt.close()

    plt.figure()
    shap.summary_plot(shap_values, X_transformed, feature_names=feature_names_final,
                       plot_type='dot', max_display=20, show=False)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/shap_beeswarm.png", dpi=130, bbox_inches='tight', facecolor='white')
    plt.close()

    plt.figure()
    shap.summary_plot(shap_values, X_transformed, feature_names=feature_names_final,
                       plot_type='bar', max_display=20, show=False)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/shap_bar.png", dpi=130, bbox_inches='tight', facecolor='white')
    plt.close()
    print("   -> Figuras SHAP guardadas: shap_summary_top20.png, shap_beeswarm.png, shap_bar.png")
except Exception as e:
    print(f"No se pudieron generar los plots SHAP ({e}).")


   -> Figuras SHAP guardadas: shap_summary_top20.png, shap_beeswarm.png, shap_bar.png


# · Validacion biologica de las features mas predictivas

Comparación de genes más importantes del SHAP con dos fuentes:
- **Open Targets** (consulta en vivo API).
- **Lista fija**: genes de riesgo genético y de expresión de IBD.

In [ ]:
# Lista de genes de riesgo genetico en IBD (de un estudio de 2017)

IBD_GENES_GWAS = {
    'NOD2', 'IL23R', 'ATG16L1', 'IL10', 'STAT3', 'TNF', 'IL6', 'IL1B', 'JAK2',
    'CARD9', 'PTPN2', 'HLA-DRB1', 'IRGM', 'TYK2', 'NKX2-3', 'ORMDL3', 'PTGER4',
    'ITLN1', 'FUT2', 'IL12B', 'IL2RA', 'TNFSF15', 'NFKB1', 'REL', 'ICOSLG',
    'SMAD3', 'ERAP2', 'GPR65', 'CARD11', 'IFIH1',
}

# Lista de genes que se sabe que se expresan mas en la mucosa inflamada de IBD

IBD_GENES_EXPRESSION = {
    'S100A8', 'S100A9', 'S100A12', 'CCL20', 'DUOX2', 'REG3A', 'REG1B',
    'MMP3', 'MMP9', 'LCN2', 'CXCL8', 'NOS2',
    # Ampliación de otros estudios (Haberman et al. 2014, J Clin Invest; Arijs et al., curación mucosa en UC)
    'DUOXA2', 'REG1A', 'MMP1', 'MMP12', 'CXCL1', 'CXCL9', 'CXCL10', 'CXCL11',
    'SAA1', 'SAA2', 'IL1B', 'CHI3L1', 'OLFM4', 'DEFA5', 'TREM1',
}
IBD_GENES_STATIC = IBD_GENES_GWAS | IBD_GENES_EXPRESSION
print(f"Lista estatica de respaldo: {len(IBD_GENES_STATIC)} genes "
      f"({len(IBD_GENES_GWAS)} susceptibilidad GWAS + {len(IBD_GENES_EXPRESSION)} expresion/inflamacion)")

def try_fetch_opentargets_ibd_genes(top_n=100):
    try:
        import requests
        # Se busca el ID de la enfermedad por nombre en vez de escribirlo fijo.
        # Open Targets cambio hace poco el formato de IDs, fijarlo podria quedar desactualizado

        search_query = '''
        query searchDisease($q: String!) {
          search(queryString: $q, entityNames: ["disease"]) {
            hits { id name entity }
          }
        }
        '''
        resp = requests.post(
            "https://api.platform.opentargets.org/api/v4/graphql",
            json={"query": search_query, "variables": {"q": "inflammatory bowel disease"}},
            timeout=15,
        )
        resp.raise_for_status()
        hits = resp.json()['data']['search']['hits']
        if not hits:
            raise ValueError("Open Targets no devolvió ninguna enfermedad para la búsqueda")
        disease_id = hits[0]['id']
        print(f"   Open Targets: ID resuelto dinámicamente = {disease_id} ({hits[0]['name']})")

        assoc_query = '''
        query associatedTargets($efoId: String!, $size: Int!) {
          disease(efoId: $efoId) {
            associatedTargets(page: {index: 0, size: $size}) {
              rows { target { approvedSymbol } score }
            }
          }
        }
        '''
        resp2 = requests.post(
            "https://api.platform.opentargets.org/api/v4/graphql",
            json={"query": assoc_query, "variables": {"efoId": disease_id, "size": top_n}},
            timeout=15,
        )
        resp2.raise_for_status()
        rows = resp2.json()['data']['disease']['associatedTargets']['rows']
        return {r['target']['approvedSymbol'] for r in rows}
    except Exception as e:
        print(f"   (Open Targets no disponible: {e} - se usa solo la lista estatica)")
        return set()

IBD_GENES_OPENTARGETS = try_fetch_opentargets_ibd_genes()
print(f"Open Targets: {len(IBD_GENES_OPENTARGETS)} genes recuperados")

IBD_GENES_ALL = IBD_GENES_STATIC | IBD_GENES_OPENTARGETS
print(f"Total combinado (unión, sin duplicados): {len(IBD_GENES_ALL)} genes")

Lista estatica de respaldo: 56 genes (30 susceptibilidad GWAS + 27 expresion/inflamacion)
   Open Targets: ID resuelto dinámicamente = MONDO_0005265 (inflammatory bowel disease)
Open Targets: 100 genes recuperados
Total combinado (unión, sin duplicados): 134 genes


In [ ]:
# Cruce de las features mas importantes del SHAP con los datos (log2FC, padj) y la lista de genes IBD

top_shap = shap_importance.head(30).copy()
top_shap['feature_clean'] = top_shap['feature'].str.replace('MISSING__', '', regex=False)
top_shap['es_indicador_ausencia'] = top_shap['feature'].str.startswith('MISSING__')

# En vez de reconstruir 'gen'/'tipo_celular' recortando el nombre de la columna, se cruza directamente por
# 'column_name', que es la clave que guardó notebook 08.

top_shap_annotated = top_shap.merge(
    meta_cols[['column_name', 'tipo_celular', 'gen', 'comparacion', 'log2FC', 'padj']],
    left_on='feature_clean', right_on='column_name', how='left'
)
top_shap_annotated['en_lista_IBD_conocida'] = top_shap_annotated['gen'].isin(IBD_GENES_ALL)

top_shap_annotated.to_csv(f"{TABLES_DIR}/top_features_biological_validation.csv", index=False)

n_known = top_shap_annotated['en_lista_IBD_conocida'].sum()
print(f"De las top 30 features por SHAP, {n_known} son genes ya conocidos en IBD ")
print(top_shap_annotated[['feature', 'gen', 'tipo_celular', 'log2FC', 'padj',
                           'en_lista_IBD_conocida', 'es_indicador_ausencia']].to_string(index=False))

if top_shap_annotated['es_indicador_ausencia'].any():
    print("\nAlgunas de las top features son indicadores de que faltaba el dato (MISSING), "
          "no expresión real.")


De las top 30 features por SHAP, 2 son genes ya conocidos en IBD 
                                             feature        gen                     tipo_celular  log2FC         padj  en_lista_IBD_conocida  es_indicador_ausencia
                        Colonocytes__DPY30__CD_vs_HC      DPY30                      Colonocytes -1.2789 1.214400e-03                  False                  False
                       Colonocytes__CYP3A4__CD_vs_HC     CYP3A4                      Colonocytes -6.6106 9.935100e-04                  False                  False
                              PC_IgA__TLE1__CD_vs_HC       TLE1                           PC IgA  2.1435 8.169400e-05                  False                  False
                     Colonocytes__APOBEC3C__CD_vs_HC   APOBEC3C                      Colonocytes -3.6922 2.359200e-03                  False                  False
                           CD4__UBXN10-AS1__UC_vs_HC UBXN10-AS1                              CD4  4.7187 6.232100e

# · Resumen final

In [ ]:
summary = {
    'n_pacientes': n_patients,
    'n_features': n_features,
    'ratio_features_pacientes': round(n_features / n_patients, 1),
    'add_indicator_activado': USE_MISSING_INDICATOR,
    'outliers_calculados': outlier_flags,
    'binario': {
        'mejor_modelo': best_model_binary,
        'metricas': results_binary,
        'permutacion_pvalue': pvalue_binary,
        'n_permutaciones': N_PERMUTATIONS,
    },
    'multiclase': {
        'mejor_modelo': best_model_multi,
        'metricas': results_multi,
        'permutacion_pvalue': pvalue_multi,
        'n_permutaciones': N_PERMUTATIONS,
    },
    'alpha_corregido_bonferroni_2_tests': ALPHA_CORRECTED,
    'decision_outliers': 'excluidos' if use_final_subset else 'mantenidos (los 18)',
}

# Predicciones por paciente, para revisar quien se clasifica peor

pred_export = pd.DataFrame({
    'paciente': features['paciente'],
    'condition_real': features['condition'],
    f'pred_binaria_{best_model_binary}': le_binary.inverse_transform(predictions_binary[best_model_binary]['y_pred']),
    f'proba_IBD_{best_model_binary}': predictions_binary[best_model_binary]['y_proba'],
    f'pred_multiclase_{best_model_multi}': le_multi.inverse_transform(predictions_multi[best_model_multi]['y_pred']),
})
pred_export.to_csv(f"{TABLES_DIR}/predicciones_por_paciente.csv", index=False)

print(f"\n  -> Guardado: 09_run_summary.json, predicciones_por_paciente.csv")


  -> Guardado: 09_run_summary.json, predicciones_por_paciente.csv
